# Character-level name generator

Build a small neural network that predicts the next character from the previous three. The notebook loads names, creates train/dev/test splits, trains the model, evaluates it, and samples new names. Run the cells from top to bottom.

## 1. Load the data

This notebook uses the public [makemore names dataset](https://github.com/karpathy/makemore/blob/master/names.txt), so it runs in a fresh Colab session without mounting Google Drive.

In [ ]:
import random
from urllib.request import urlopen

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

DATA_URL = 'https://raw.githubusercontent.com/karpathy/makemore/master/names.txt'
with urlopen(DATA_URL, timeout=30) as response:
    words = response.read().decode('utf-8').split()

if not words:
    raise ValueError('The names dataset is empty')

print(f'Loaded {len(words):,} names. Examples: {words[:10]}')

## 2. Encode characters and build datasets

The `.` token marks the beginning and end of a name. Each example contains three context characters and the character that follows.

In [ ]:
block_size = 3
chars = sorted(set(''.join(words)))
stoi = {char: index + 1 for index, char in enumerate(chars)}
stoi['.'] = 0
itos = {index: char for char, index in stoi.items()}
vocab_size = len(stoi)

def build_dataset(items):
    contexts, targets = [], []
    for word in items:
        context = [0] * block_size
        for char in word + '.':
            target = stoi[char]
            contexts.append(context)
            targets.append(target)
            context = context[1:] + [target]
    return torch.tensor(contexts, dtype=torch.long), torch.tensor(targets, dtype=torch.long)

shuffled_words = words.copy()
random.Random(42).shuffle(shuffled_words)
n1 = int(0.8 * len(shuffled_words))
n2 = int(0.9 * len(shuffled_words))
Xtr, Ytr = build_dataset(shuffled_words[:n1])
Xdev, Ydev = build_dataset(shuffled_words[n1:n2])
Xte, Yte = build_dataset(shuffled_words[n2:])

print(f'Vocabulary: {vocab_size} tokens; train/dev/test examples: '
      f'{len(Xtr):,}/{len(Xdev):,}/{len(Xte):,}')

## 3. Understand embeddings

An embedding matrix stores one learned vector per token. Direct indexing selects the same row as multiplying by a one-hot vector. The small example below is separate from the model trained later.

In [ ]:
demo_embeddings = torch.randn((vocab_size, 2), generator=torch.Generator().manual_seed(42))
token = stoi[chars[0]]
one_hot = F.one_hot(torch.tensor(token), num_classes=vocab_size).float()
print('Direct lookup:', demo_embeddings[token])
print('One-hot multiplication matches:', torch.allclose(one_hot @ demo_embeddings,
                                                        demo_embeddings[token]))
print('Batch embedding shape:', demo_embeddings[Xtr[:32]].shape)

## 4. Initialize and train the model

Three 10-dimensional embeddings feed a 200-unit hidden layer. Cross-entropy compares predicted next-character scores with the true targets.

In [ ]:
embedding_dim = 10
hidden_size = 200
generator = torch.Generator().manual_seed(2147483647)

C = torch.randn((vocab_size, embedding_dim), generator=generator)
W1 = torch.randn((block_size * embedding_dim, hidden_size), generator=generator)
b1 = torch.randn(hidden_size, generator=generator)
W2 = torch.randn((hidden_size, vocab_size), generator=generator)
b2 = torch.randn(vocab_size, generator=generator)
parameters = [C, W1, b1, W2, b2]
for parameter in parameters:
    parameter.requires_grad_(True)

print(f'Model parameters: {sum(p.numel() for p in parameters):,}')

In [ ]:
steps = 50_000
batch_size = 32
learning_rate = 0.01
loss_history = []

for step in range(steps):
    batch = torch.randint(0, len(Xtr), (batch_size,), generator=generator)
    embeddings = C[Xtr[batch]]
    hidden = torch.tanh(embeddings.reshape(batch_size, -1) @ W1 + b1)
    logits = hidden @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[batch])

    for parameter in parameters:
        parameter.grad = None
    loss.backward()
    with torch.no_grad():
        for parameter in parameters:
            parameter -= learning_rate * parameter.grad

    loss_history.append(loss.item())

print(f'Final batch loss: {loss_history[-1]:.4f}')

## 5. Evaluate

Training loss measures fit to seen examples. Dev loss helps tune the model; test loss is reserved for a final check.

In [ ]:
def dataset_loss(inputs, targets):
    with torch.no_grad():
        embeddings = C[inputs]
        hidden = torch.tanh(embeddings.reshape(len(inputs), -1) @ W1 + b1)
        return F.cross_entropy(hidden @ W2 + b2, targets).item()

for label, inputs, targets in [('Train', Xtr, Ytr),
                               ('Dev', Xdev, Ydev),
                               ('Test', Xte, Yte)]:
    print(f'{label} loss: {dataset_loss(inputs, targets):.4f}')

plt.figure(figsize=(8, 4))
plt.plot(loss_history)
plt.xlabel('Training step')
plt.ylabel('Batch cross-entropy loss')
plt.title('Training loss')
plt.grid(alpha=0.3)
plt.show()

## 6. Sample names

Generate one character at a time until the end token is sampled.

In [ ]:
sample_generator = torch.Generator().manual_seed(2147483647)
with torch.no_grad():
    for _ in range(20):
        context = [0] * block_size
        generated = []
        while True:
            embeddings = C[torch.tensor([context])]
            hidden = torch.tanh(embeddings.reshape(1, -1) @ W1 + b1)
            probabilities = F.softmax(hidden @ W2 + b2, dim=1)
            token = torch.multinomial(probabilities, 1, generator=sample_generator).item()
            if token == 0:
                break
            generated.append(itos[token])
            context = context[1:] + [token]
        print(''.join(generated))